# Task 2 · Overfitting
The Task 1 CNN against a larger CNN (1.5× wider convolutions, 2× wider FC layer, 2.9× the parameters), both trained with
the same 5 folds and schedule as in Task 1; then the large CNN with dropout or with weight decay.

In [ ]:
from hw3 import *

B = json.load(open(BEST))['cnn']
large = dict(model=partial(CNN, 48, 512), lr=B['lr'])
S = {'base': dict(model=partial(CNN, B['arch']), lr=B['lr']), 'large': large,
     'dropout': large | dict(model=partial(CNN, 48, 512, .5)), 'wd': large | dict(wd=.5)}
key = {'base': B['key'], 'large': 'large', 'dropout': 'large_dropout', 'wd': 'large_wd'}
R = cv({key[k]: S[k] for k in S})
R = {k: R[key[k]] for k in S}
diagram('arch_large', [cnn_layers(B['arch']), cnn_layers(48, 512)])

Generalization loss $GL(t) = 100\,(L_{val}(t) / \min_{s \le t} L_{val}(s) - 1)$: how far, in %, the validation loss is
above the best value seen so far. Overfitting starts at the first epoch from which $GL$ stays above 5% to the end.

In [ ]:
ALPHA = 5
gl = lambda r: 100 * (np.asarray(r['val_loss']) / np.minimum.accumulate(np.asarray(r['val_loss'])) - 1)


def onset(r):
    above = gl(r) > ALPHA
    return len(above) - np.argmin(above[::-1]) + 1 if above[-1] else np.nan


ons = {k: np.array([onset(r) for r in R[k]], float) for k in R}
pd.DataFrame(ons, index=[f'fold {f + 1}' for f in range(FOLDS)])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4))
G = {'base CNN': ('base', COL['cnn']), 'large CNN': ('large', COL['large'])}
curves({k: (R[n], c) for k, (n, c) in G.items()}, axes[:2])
for k, (n, c) in G.items():
    band(axes[2], [dict(gl=gl(r)) for r in R[n]], 'gl', c, k)
    if np.isfinite(ons[n]).all():
        for ax in axes:
            ax.axvline(ons[n].mean(), color=c, ls=':', lw=1)
axes[2].axhline(ALPHA, color=INK, ls='--', lw=.8)
axes[2].set(xlabel='epoch', ylabel='generalization loss GL (%)')
save(fig, 'overfit')

In [ ]:
G = {'large CNN': ('large', COL['large']), '+ dropout 0.5': ('dropout', COL['dropout']),
     '+ weight decay 0.5': ('wd', COL['wd'])}
fig, axes = plt.subplots(2, 2, figsize=(7.2, 4.2))
curves({k: (R[n], c) for k, (n, c) in G.items()}, axes[0])
for k, (n, c) in G.items():
    band(axes[1, 0], R[n], 'wnorm', c, k)
    w = load(key[n], S[n]['model'])[5].weight.detach().flatten().numpy()
    axes[1, 1].hist(w, 120, histtype='step', color=c, density=True)
axes[1, 0].set(xlabel='epoch', ylabel='norm of all weights')
axes[1, 1].set(xlabel='FC-1 weights (3072×512, fold 1)', ylabel='density', yscale='log')
save(fig, 'regularize')

In [ ]:
on = lambda k: ms(ons[k], 1) if np.isfinite(ons[k]).all() else f'{np.isfinite(ons[k]).sum()} of {FOLDS} folds'
T = pd.DataFrame({k: {'params (M)': f"{R[k][0]['params'] / 1e6:.2f}", 'train acc': ms(last(R[k], 'train_acc')),
                      'val acc': ms(last(R[k], 'val_acc')), 'test acc': ms(last(R[k], 'test_acc')),
                      'gap (pp)': ms(last(R[k], 'train_acc') - last(R[k], 'val_acc')),
                      'best val loss': ms([r['val_loss'].min().item() for r in R[k]], 1, 3),
                      'final val loss': ms(last(R[k], 'val_loss'), 1, 3), 'onset epoch': on(k)} for k in R}).T
rows('t2_table', T.set_axis(['Base CNN', 'Large CNN', '+ dropout', '+ weight decay']))
tex('t2', glAlpha=ALPHA, largeDiff=pdiff(R['large'], R['base']), dropoutDiff=pdiff(R['dropout'], R['large']),
    wdDiff=pdiff(R['wd'], R['large']), **{k + n: T[c][k] for k in R for n, c in (
        ('Onset', 'onset epoch'), ('Gap', 'gap (pp)'), ('Test', 'test acc'), ('Loss', 'final val loss'))})
T